<a href="https://colab.research.google.com/github/nazndev/visibility-gated-clip-reid/blob/main/notebooks/colab_iteration2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Iteration 2

Cell names, from top to bottom:

1. **Cell 1 — clone.** Run this after every runtime restart.
2. **Cell 2 — GPU.** It must print `nvidia device True`.
3. **Cell 3 — install.**
4. **Cell 4 — Drive, dataset, and checkpoint.**
5. **Cell 5 — plan.** You can skip this. The experiment does not need it.
6. **Cell 6 — experiment.** This is the only cell to rerun for the next seed. Seed 11 is already saved. Set `SEED` to `"23"` next.
7. **Cell 7 — saved scores.**


In [1]:
# Cell 1 — clone. Run this again after a runtime restart.
import os
import re
import subprocess
from pathlib import Path

REVISION = ""  # blank locks origin/main; or paste a 40-character commit
REPO = "https://github.com/nazndev/visibility-gated-clip-reid.git"
ROOT = Path("/content/lvlm-person-reid")

if not (ROOT / "src" / "reidvlm").is_dir():
    subprocess.check_call(["git", "clone", REPO, str(ROOT)])
subprocess.check_call(["git", "fetch", "origin"], cwd=ROOT)
if not re.fullmatch(r"[0-9a-f]{40}", REVISION):
    REVISION = subprocess.check_output(["git", "rev-parse", "origin/main"], cwd=ROOT, text=True).strip()
if not re.fullmatch(r"[0-9a-f]{40}", REVISION):
    raise SystemExit("origin/main is not a 40-character commit.")
subprocess.check_call(["git", "checkout", "--force", REVISION], cwd=ROOT)
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
if head != REVISION:
    raise SystemExit(f"HEAD {head} is not {REVISION}")
os.chdir(ROOT)
print(head)


03db1a1d35d55ddbe6c7e74e8f709884c53b4910


In [2]:
# Cell 2 — GPU check. Continue only if this prints nvidia device True.
import os

print("nvidia device", os.path.exists("/dev/nvidia0"))
if not os.path.exists("/dev/nvidia0"):
    raise RuntimeError("Select a GPU runtime, then run this cell again.")


nvidia device True


In [3]:
# Cell 3 — install the project.
import subprocess
from pathlib import Path

ROOT = Path("/content/lvlm-person-reid")
subprocess.check_call(["pip", "install", "-q", "-r", "requirements-colab.txt"], cwd=ROOT)
subprocess.check_call(["pip", "install", "-q", "-e", "."], cwd=ROOT)
import torch
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no cuda"
print(torch.__version__, torch.cuda.is_available(), name)


2.15.0+cu132 True Tesla T4


In [4]:
# Cell 4 — mount Drive and download the dataset and checkpoint if they are missing.
import subprocess
from pathlib import Path
from google.colab import drive

ROOT = Path("/content/lvlm-person-reid")
DRIVE = Path("/content/drive/MyDrive/visibility-gated-reid/iteration2")
drive.mount("/content/drive")
DRIVE.mkdir(parents=True, exist_ok=True)
if not (ROOT / "data/Market-1501-v15.09.15/bounding_box_train").is_dir():
    subprocess.check_call(["python", "scripts/01_download_dataset.py"], cwd=ROOT)
if not (ROOT / "checkpoints/Market1501_clipreid_ViT-B-16_60.pth").is_file():
    subprocess.check_call(["python", "scripts/02_download_model.py"], cwd=ROOT)
print(DRIVE)


Mounted at /content/drive
/content/drive/MyDrive/visibility-gated-reid/iteration2


In [1]:
# Cell 5 — write the plan and validation manifest. Safe to skip. The experiment cell does not need this.
import subprocess
from pathlib import Path

ROOT = Path("/content/lvlm-person-reid")
DRIVE = Path("/content/drive/MyDrive/visibility-gated-reid/iteration2")
subprocess.check_call(["python", "-m", "reidvlm", "prune-plan", "--output", str(DRIVE / "iteration2_plan.json")], cwd=ROOT)
subprocess.check_call(["python", "-m", "reidvlm", "prune-manifest", "--config", "configs/market1501.yaml", "--output", str(DRIVE / "validation_manifest.json")], cwd=ROOT)
print("plan and manifest written")


FileNotFoundError: [Errno 2] No such file or directory: PosixPath('/content/lvlm-person-reid')

In [6]:
# Cell 6 — experiment. Change SEED, then run only this cell. Seed 11 is already done; use 23 next. Do not start a second copy while this is running.
STRATEGY = "random"  # none, cls_attention, random
BLOCK = "6"
KEEP = "64"
SEED = "11"  # required for random: 11, 23, 37, 53, or 71
SPLIT = "validation"  # validation uses training images; test is the official split
BACKEND = "reference"  # reference or cls_row

import subprocess
from pathlib import Path

ROOT = Path("/content/lvlm-person-reid")
DRIVE = Path("/content/drive/MyDrive/visibility-gated-reid/iteration2")
if STRATEGY == "random" and SEED not in {"11", "23", "37", "53", "71"}:
    raise SystemExit("Random selection needs SEED set to 11, 23, 37, 53, or 71.")
name = f"{SPLIT}_{STRATEGY}_b{BLOCK}_k{KEEP}_{BACKEND}" + (f"_s{SEED}" if STRATEGY == "random" else "")
command = [
    "python", "-m", "reidvlm", "prune-embed",
    "--config", "configs/market1501.yaml",
    "--split", SPLIT,
    "--strategy", STRATEGY,
    "--prune-block", BLOCK,
    "--keep", KEEP,
    "--backend", BACKEND,
    "--batch-size", "32",
    "--run-dir", str(DRIVE / name),
    "--output", str(DRIVE / f"{name}.json"),
]
if STRATEGY == "random":
    command.extend(["--seed", SEED])
print(" ".join(command))
subprocess.check_call(command, cwd=ROOT)


python -m reidvlm prune-embed --config configs/market1501.yaml --split validation --strategy random --prune-block 6 --keep 64 --backend reference --batch-size 32 --run-dir /content/drive/MyDrive/visibility-gated-reid/iteration2/validation_random_b6_k64_reference_s11 --output /content/drive/MyDrive/visibility-gated-reid/iteration2/validation_random_b6_k64_reference_s11.json --seed 11


0

In [7]:
# Cell 7 — list the score files already saved on Drive.
import json
from pathlib import Path

DRIVE = Path("/content/drive/MyDrive/visibility-gated-reid/iteration2")
rows = []
for path in sorted(DRIVE.glob("*.json")):
    if path.name in {"iteration2_plan.json", "validation_manifest.json"}:
        continue
    payload = json.loads(path.read_text())
    scores = payload.get("scores", {})
    rows.append({
        "file": path.name,
        "label": payload.get("label"),
        "mAP": scores.get("mAP"),
        "rank1": scores.get("rank1"),
        "kreciprocal": payload.get("kreciprocal"),
    })
print(json.dumps(rows, indent=2))


[
  {
    "file": "validation_random_b6_k64_reference_s11.json",
    "label": "full",
    "mAP": 0.7894766290099927,
    "rank1": 0.8948069241011984,
    "kreciprocal": "not_run"
  }
]
